In [4]:
import sqlite3 as sql

In [5]:
db = sql.connect("tea.db")

In [6]:
cursor = db.cursor()

In [25]:
cursor.execute("""CREATE TABLE IF NOT EXISTS Tea (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name VARCHAR(100) UNIQUE NOT NULL,
        description TEXT NOT NULL,
        is_tea BOOLEAN NOT NULL,
        has_caffeine BOOLEAN NOT NULL,
        url TEXT UNIQUE NOT NULL,
        img_url TEXT UNIQUE NOT NULL,
        price DECIMAL(10, 2) NOT NULL,
        warning TEXT,
        cook TEXT NOT NULL);
    """)

In [26]:
cursor.execute("""CREATE TABLE IF NOT EXISTS Herb (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name VARCHAR(20) NOT NULL UNIQUE,
        description TEXT NOT NULL UNIQUE,
        family_name VARCHAR(30),
        part_used VARCHAR(20) NOT NULL,
        image_url TEXT NOT NULL);
    """)

In [27]:
cursor.execute("""CREATE TABLE IF NOT EXISTS Ingredients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        tea INTEGER NOT NULL,
        herb VARCHAR(20) NOT NULL,
        FOREIGN KEY(tea) REFERENCES Tea(id),
        FOREIGN KEY(herb) REFERENCES Herb(name)
        );
    """)

In [28]:
cursor.execute("""CREATE TABLE IF NOT EXISTS Faq (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        question TEXT NOT NULL,
        answer TEXT NOT NULL,
        tea INTEGER NOT NULL,
        FOREIGN KEY(tea) REFERENCES Tea(id)
    );""")

In [29]:
cursor.execute("""CREATE TABLE IF NOT EXISTS Benefit (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        benefit TEXT NOT NULL
    );""")

In [30]:
cursor.execute("""CREATE TABLE IF NOT EXISTS Herb_benefit (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        benefit INTEGER NOT NULL,
        herb INTEGER NOT NULL,
        FOREIGN KEY(benefit) REFERENCES Benefit(id),
        FOREIGN KEY(Herb) REFERENCES Herb(id)
    );""")

In [31]:
db.commit()

In [2]:
import pandas as pd

herbs_df = pd.read_csv("herbs.csv")
faqs_df = pd.read_csv("faqs.csv")

print(len(herbs_df), "linhas de ervas |", len(faqs_df), "FAQs")

154 linhas de ervas | 223 FAQs


In [9]:
db.execute("PRAGMA foreign_keys = ON;")

cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS ux_ingredients   ON Ingredients (tea, herb);")
cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS ux_faq           ON Faq (tea, question);")
cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS ux_benefit       ON Benefit (benefit);")
cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS ux_herb_benefit  ON Herb_benefit (herb, benefit);")

db.commit()

In [7]:
tea_ids = dict(cursor.execute("SELECT name, id FROM Tea;").fetchall())
herb_ids = dict(cursor.execute("SELECT name, id FROM Herb;").fetchall())

print(len(tea_ids), "chas |", len(herb_ids), "ervas")

51 chas | 44 ervas


In [12]:
pares = [
    (tea_ids[row.product_name], row.herb_name)
    for row in herbs_df.itertuples()
    if row.product_name in tea_ids
]

cursor.executemany("INSERT OR IGNORE INTO Ingredients (tea, herb) VALUES (?, ?);", pares)
db.commit()

print("Ingredients:", cursor.execute("SELECT COUNT(*) FROM Ingredients;").fetchone()[0])

Ingredients: 154


In [13]:
faq_rows = [
    (row.question, row.answer, tea_ids[row.product_name])
    for row in faqs_df.itertuples()
    if row.product_name in tea_ids
]

cursor.executemany("INSERT OR IGNORE INTO Faq (question, answer, tea) VALUES (?, ?, ?);", faq_rows)
db.commit()

print("Faq:", cursor.execute("SELECT COUNT(*) FROM Faq;").fetchone()[0])

Faq: 223


In [9]:
def split_benefits(valor):
    if pd.isna(valor):
        return []
    return [b.strip() for b in str(valor).split(";") if b.strip()]


herb_para_beneficios = {
    row.herb_name: split_benefits(row.herb_benefits)
    for row in herbs_df.drop_duplicates(subset=["herb_name"]).itertuples()
}

todos = sorted({b for lista in herb_para_beneficios.values() for b in lista})

cursor.executemany("INSERT OR IGNORE INTO Benefit (benefit) VALUES (?);", [(b,) for b in todos])
db.commit()

benefit_ids = dict(cursor.execute("SELECT benefit, id FROM Benefit;").fetchall())
print(len(benefit_ids), "beneficios:", todos)

33 beneficios: ['Adds color to your cup', 'Caffeine-free beverage + tea or coffee substitute', 'Detox', 'Digestion', 'Endurance', 'Energy', 'Heart Health', 'Immunity', 'Joint Health', 'Lactation', 'Laxative', 'Libido Support', 'Mental Focus', 'Mood Support', 'Nausea', 'Pre & Postnatal', 'Relaxation', 'Respiratory Health', 'Seasonal Care', 'Skin Health', 'Sleep Support', 'Stress Relief', 'Throat Health', 'Water Retention', "Women's Cycle"]


In [15]:
ligacoes = [
    (benefit_ids[b], herb_ids[nome])
    for nome, lista in herb_para_beneficios.items()
    if nome in herb_ids
    for b in lista
]

cursor.executemany("INSERT OR IGNORE INTO Herb_benefit (benefit, herb) VALUES (?, ?);", ligacoes)
db.commit()

print("Herb_benefit:", cursor.execute("SELECT COUNT(*) FROM Herb_benefit;").fetchone()[0])

Herb_benefit: 95


In [16]:
verificacoes = {
    "Ingredients orfaos": """SELECT COUNT(*) FROM Ingredients i
         LEFT JOIN Tea t ON t.id = i.tea
         LEFT JOIN Herb h ON h.name = i.herb
        WHERE t.id IS NULL OR h.id IS NULL""",
    "Faq orfaos": """SELECT COUNT(*) FROM Faq f
         LEFT JOIN Tea t ON t.id = f.tea WHERE t.id IS NULL""",
    "chas sem ervas": """SELECT COUNT(*) FROM Tea t
        WHERE NOT EXISTS (SELECT 1 FROM Ingredients i WHERE i.tea = t.id)""",
    "ervas orfas (sem cha)": """SELECT COUNT(*) FROM Herb h
        WHERE NOT EXISTS (SELECT 1 FROM Ingredients i WHERE i.herb = h.name)""",
}

for etiqueta, sql_query in verificacoes.items():
    print(f"{etiqueta:24}: {cursor.execute(sql_query).fetchone()[0]}")

for tabela in ["Tea", "Herb", "Ingredients", "Faq", "Benefit", "Herb_benefit"]:
    print(f"{tabela:14}", cursor.execute(f"SELECT COUNT(*) FROM {tabela};").fetchone()[0])

Ingredients orfaos      : 0
Faq orfaos              : 0
chas sem ervas          : 0
ervas orfas (sem cha)   : 0
Tea            51
Herb           44
Ingredients    154
Faq            223
Benefit        31
Herb_benefit   95


In [17]:
for nome, beneficios in cursor.execute("""
    SELECT h.name, GROUP_CONCAT(b.benefit, ', ')
      FROM Ingredients i
      JOIN Tea  t  ON t.id   = i.tea
      JOIN Herb h  ON h.name = i.herb
      LEFT JOIN Herb_benefit hb ON hb.herb = h.id
      LEFT JOIN Benefit b       ON b.id    = hb.benefit
     WHERE t.name LIKE 'Nighty Night Extra%'
     GROUP BY h.id;""").fetchall():
    print(f"{nome:16} -> {beneficios}")

Lemon Balm       -> Digestion, Good Mood, Relaxation
Valerian         -> Relaxation, Sleep
Passionflower    -> Relaxation, Sleep, Stress Relief
Licorice         -> Digestive, Respiratory, Throat Health
Peppermint       -> Digestion


In [18]:
colunas = [c[1] for c in cursor.execute("PRAGMA table_info(Tea);").fetchall()]

if "benefit_headline" not in colunas:
    cursor.execute("ALTER TABLE Tea ADD COLUMN benefit_headline TEXT;")

cursor.execute("""CREATE TABLE IF NOT EXISTS Tea_benefit (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        tea INTEGER NOT NULL,
        benefit INTEGER NOT NULL,
        FOREIGN KEY(tea) REFERENCES Tea(id),
        FOREIGN KEY(benefit) REFERENCES Benefit(id)
    );""")
cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS ux_tea_benefit ON Tea_benefit (tea, benefit);")

db.commit()

In [19]:
teas_df = pd.read_json("teas.json")

# 1. headline declarada pelo fabricante
cursor.executemany(
    "UPDATE Tea SET benefit_headline = ? WHERE name = ?;",
    [(row.benefit_headline, row["name"]) for _, row in teas_df.iterrows()],
)

# 2. beneficios do produto — vem das tags da loja, filtrando as que
#    nao sao beneficios (certificacoes, sabores, tags internas)
NAO_BENEFICIO = {
    "USDA Organic", "Fair Trade", "Fair for Life", "FairWild", "Panda Friendly",
    "Berry", "Citrus", "Earthy", "Floral", "Mint", "Spice", "Naturally Sweet",
    "Tart and Tangy", "Green Tea", "Ginger", "Licorice", "Lavender", "Dandelion",
    "Everyday Herbals", "Daily Wellness", "Probiotics", "infront",
    "tiered_discount", "YGroup_SECinnamonTension",
}

pares_tb = []
for _, row in teas_df.iterrows():
    tea_id = tea_ids.get(row["name"])
    if tea_id is None:
        continue
    for tag in row["shopify_tags"]:
        if tag in NAO_BENEFICIO:
            continue
        if tag not in benefit_ids:                      # beneficio novo
            cursor.execute("INSERT OR IGNORE INTO Benefit (benefit) VALUES (?);", (tag,))
            benefit_ids = dict(cursor.execute("SELECT benefit, id FROM Benefit;").fetchall())
        pares_tb.append((tea_id, benefit_ids[tag]))

cursor.executemany("INSERT OR IGNORE INTO Tea_benefit (tea, benefit) VALUES (?, ?);", pares_tb)
db.commit()

print("Tea_benefit:", cursor.execute("SELECT COUNT(*) FROM Tea_benefit;").fetchone()[0])

Tea_benefit: 107


In [20]:
for nome, declarados, uniao in cursor.execute("""
    SELECT t.name,
           (SELECT COUNT(*) FROM Tea_benefit tb WHERE tb.tea = t.id),
           (SELECT COUNT(DISTINCT hb.benefit)
              FROM Ingredients i
              JOIN Herb h ON h.name = i.herb
              JOIN Herb_benefit hb ON hb.herb = h.id
             WHERE i.tea = t.id)
      FROM Tea t
     ORDER BY 3 - 2 DESC
     LIMIT 10;""").fetchall():
    print(f"{nome[:38]:40} declarados={declarados}  uniao_ervas={uniao}")

Belly Comfort® Peppermint Tea            declarados=1  uniao_ervas=4
Breathe Easy® Tea                        declarados=2  uniao_ervas=8
Chamomile & Lavender Tea                 declarados=2  uniao_ervas=5
Chamomile Tea                            declarados=4  uniao_ervas=4
Cold Care P.M.® Tea                      declarados=3  uniao_ervas=9
Cup of Calm® Tea                         declarados=2  uniao_ervas=7
Dandelion Leaf & Root Tea                declarados=3  uniao_ervas=3
Echinacea Plus® Elderberry Tea           declarados=2  uniao_ervas=8
Echinacea Plus® Tea                      declarados=2  uniao_ervas=4
EveryDay Detox® Hibiscus & Schisandra    declarados=1  uniao_ervas=9


In [21]:
db.commit()
db.close()